<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/04_analisis_resultados/L4S_10_sintesis_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L4S-10 — Síntesis Final del Proyecto
## Detección de Deslizamientos — Landslide4Sense

Este notebook consolida **todos** los resultados del proyecto en un único lugar:

| Fase | Protocolo | Modelos |
|---|---|---|
| **Fase 1** | 2-Fold · ~2 000 muestras | Clásicos, ResNet-50, EfficientNet-B4, U-Net |
| **Fase 2** | 5-Fold · dataset completo (3 799 muestras) | Clásicos, ResNet-50, EfficientNet-B4, U-Net |

**Secciones:**
1. Configuración y carga de resultados
2. Tabla maestra comparativa
3. Evolución Fase 1 → Fase 2
4. Variabilidad por fold (box plots)
5. Tests estadísticos (Friedman + Wilcoxon)
6. Benchmarking vs literatura
7. Transferibilidad a Colombia
8. Gap analysis (radar chart)
9. Resumen ejecutivo

> **Prerequisitos:** haber corrido los notebooks 03–09 y transferibilidad_01.

In [ ]:
# ── Celda 0: Entorno ─────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from scipy import stats

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

DRIVE_PATH = '/content/drive/MyDrive/Landslide4Sense'
ROOT = Path(DRIVE_PATH)
OUT_DIR = ROOT / 'results' / 'sintesis_final'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def load_json(rel, silent=False):
    p = ROOT / rel
    if p.exists():
        with open(p) as f: return json.load(f)
    if not silent:
        print(f'  ⚠️  No encontrado: {p.relative_to(ROOT)}')
    return None

print(f'✅ Drive montado. Salida: {OUT_DIR}')

---
## 1. Carga de resultados — Fase 1 (2-Fold) y Fase 2 (5-Fold)

In [ ]:
# ── Celda 1: Carga de todos los JSONs ────────────────────────────────────────

# ── FASE 1 (2-Fold, notebooks 03-06) ─────────────────────────────────────────
print('Cargando Fase 1 (2-Fold)...')
_cls1  = load_json('results/classical_baselines/comparison_summary.json')
_rn1   = load_json('results/resnet50/kfold_summary.json')
_eff1  = load_json('results/efficientnet_b4/kfold_summary.json')
_unet1 = load_json('results/unet_resnet34/kfold_summary.json')

# ── FASE 2 (5-Fold, notebooks L4S_01-04) ─────────────────────────────────────
print('Cargando Fase 2 (5-Fold)...')
_cls5  = load_json('results/comparable_literature/classicos_5fold/kfold5_summary.json')
_rn5   = load_json('results/comparable_literature/resnet50_5fold/kfold5_summary.json')
# Nota: algunos notebooks usaron 'literatura' en lugar de 'literature'
_eff5  = load_json('results/comparable_literatura/efficientnet_5fold/kfold5_summary.json')
if _eff5 is None:
    _eff5 = load_json('results/comparable_literature/efficientnet_5fold/kfold5_summary.json', silent=True)

_unet5_folds = []
for k in range(1, 6):
    d = load_json(f'results/comparable_literatura/unet_5fold/fold{k}_results.json', silent=True)
    if d is None:
        d = load_json(f'results/comparable_literature/unet_5fold/fold{k}_results.json', silent=True)
    if d: _unet5_folds.append(d)
if not _unet5_folds:
    print('  ⚠️  No encontrado: unet_5fold/fold*_results.json')

# ── Transferibilidad Colombia ─────────────────────────────────────────────────
print('Cargando resultados Colombia...')
_col = load_json('results/transferibilidad_01/ablacion_resultados.json', silent=True)

print(f'\nResumen de carga:')
print(f'  Fase 1 — Clásicos: {"✅" if _cls1 else "❌"}  ResNet-50: {"✅" if _rn1 else "❌"}  EfficientNet: {"✅" if _eff1 else "❌"}  U-Net: {"✅" if _unet1 else "❌"}')
print(f'  Fase 2 — Clásicos: {"✅" if _cls5 else "❌"}  ResNet-50: {"✅" if _rn5 else "❌"}  EfficientNet: {"✅" if _eff5 else "❌"}  U-Net ({len(_unet5_folds)} folds)')
print(f'  Colombia          — {"✅" if _col else "❌ (opcional)"}')

In [ ]:
# ── Celda 2: Construir tablas FASE1 y FASE2 ──────────────────────────────────

COLORES = {
    'Logistic Regression': '#6EE7B7',
    'SVM (RBF)':           '#34D399',
    'Random Forest':       '#059669',
    'ResNet-50':           '#818CF8',
    'EfficientNet-B4':     '#A78BFA',
    'U-Net ResNet-34':     '#F59E0B',
}
ORDEN = list(COLORES.keys())

def _build_fase1():
    rows = []
    if _cls1:
        for m in _cls1.get('models', []):
            rows.append({
                'nombre': m['name'], 'tipo': 'Clásico', 'nivel': 'patch',
                'f1': m.get('mean_f1', np.nan),
                'std': m.get('std_f1', 0.0),
                'auc': m.get('mean_auc_roc', np.nan),
                'folds_f1': [f['f1'] for f in m.get('folds', [])],
                'n_folds': _cls1.get('n_folds', 2),
            })
    for label, data, key in [
        ('ResNet-50',       _rn1,   'f1_thr05'),
        ('EfficientNet-B4', _eff1,  'f1_thr05'),
    ]:
        if data:
            ag = data.get('aggregate', {})
            rows.append({
                'nombre': label, 'tipo': 'Deep Learning', 'nivel': 'patch',
                'f1': ag.get('mean_f1_thr05', ag.get('mean_f1', np.nan)),
                'std': ag.get('std_f1_thr05',  ag.get('std_f1', 0.0)),
                'auc': ag.get('mean_auc_roc', np.nan),
                'folds_f1': [f.get(key, np.nan) for f in data.get('folds', [])],
                'n_folds': len(data.get('folds', [])),
            })
    if _unet1:
        ag = _unet1.get('aggregate', _unet1)
        rows.append({
            'nombre': 'U-Net ResNet-34', 'tipo': 'Deep Learning', 'nivel': 'píxel',
            'f1': ag.get('mean_f1_thr05', ag.get('mean_f1', np.nan)),
            'std': ag.get('std_f1_thr05',  ag.get('std_f1', 0.0)),
            'auc': ag.get('mean_auc_roc', np.nan),
            'folds_f1': [f.get('f1_pixel_thr05', f.get('f1_thr05', np.nan)) for f in _unet1.get('folds', [])],
            'n_folds': len(_unet1.get('folds', [])),
        })
    return rows

def _build_fase2():
    rows = []
    if _cls5:
        for key, nombre in [('logistic_regression','Logistic Regression'),
                             ('svm_rbf','SVM (RBF)'),
                             ('random_forest','Random Forest')]:
            d = _cls5['models'].get(key, {})
            if d:
                rows.append({
                    'nombre': nombre, 'tipo': 'Clásico', 'nivel': 'patch',
                    'f1': d.get('mean_f1', np.nan),
                    'std': d.get('std_f1', 0.0),
                    'auc': d.get('mean_auc_roc', np.nan),
                    'folds_f1': [f['f1'] for f in d.get('folds', [])],
                    'n_folds': 5,
                })
    for label, data in [('ResNet-50', _rn5), ('EfficientNet-B4', _eff5)]:
        if data:
            ag = data.get('aggregate', {})
            rows.append({
                'nombre': label, 'tipo': 'Deep Learning', 'nivel': 'patch',
                'f1': ag.get('mean_f1_thr05', ag.get('mean_f1', np.nan)),
                'std': ag.get('std_f1_thr05',  ag.get('std_f1', 0.0)),
                'auc': ag.get('mean_auc_roc', np.nan),
                'folds_f1': [f.get('f1_thr05', np.nan) for f in data.get('folds', [])],
                'n_folds': 5,
            })
    if _unet5_folds:
        ff = [r.get('f1_pixel_thr05', r.get('f1_thr05', np.nan)) for r in _unet5_folds]
        rows.append({
            'nombre': 'U-Net ResNet-34', 'tipo': 'Deep Learning', 'nivel': 'píxel',
            'f1': float(np.nanmean(ff)),
            'std': float(np.nanstd(ff)),
            'auc': float(np.nanmean([r.get('auc_roc', np.nan) for r in _unet5_folds])),
            'folds_f1': ff,
            'n_folds': len(_unet5_folds),
        })
    return rows

FASE1 = _build_fase1()
FASE2 = _build_fase2()

print(f'Fase 1: {len(FASE1)} modelos cargados')
print(f'Fase 2: {len(FASE2)} modelos cargados')

---
## 2. Tabla maestra comparativa

In [ ]:
# ── Celda 3: Tabla maestra ────────────────────────────────────────────────────
f1_f1 = {m['nombre']: m['f1']  for m in FASE1}
f1_f2 = {m['nombre']: m['f1']  for m in FASE2}
std_f1 = {m['nombre']: m['std'] for m in FASE1}
std_f2 = {m['nombre']: m['std'] for m in FASE2}

SEP = '=' * 82
print(SEP)
print(f'  {"Modelo":<22} {"Tipo":<15} {"Nivel":<7}'
      f' {"F1 Fase1":>9} {"±":>2} {"F1 Fase2":>9} {"±":>2} {"Δ pp":>7}')
print(SEP)

for nombre in ORDEN:
    f1  = f1_f1.get(nombre)
    f2  = f1_f2.get(nombre)
    s1  = std_f1.get(nombre, 0)
    s2  = std_f2.get(nombre, 0)
    tipo = next((m['tipo']  for m in (FASE1+FASE2) if m['nombre']==nombre), '—')
    nivel= next((m['nivel'] for m in (FASE1+FASE2) if m['nombre']==nombre), '—')
    if f1 is None and f2 is None: continue
    f1_s = f'{f1:.4f}' if f1 and not np.isnan(f1) else '   —   '
    s1_s = f'{s1:.4f}' if f1 and not np.isnan(f1) else '      '
    f2_s = f'{f2:.4f}' if f2 and not np.isnan(f2) else '   —   '
    s2_s = f'{s2:.4f}' if f2 and not np.isnan(f2) else '      '
    if f1 and f2 and not np.isnan(f1) and not np.isnan(f2):
        delta = (f2 - f1) * 100
        d_s = f'{delta:+.1f}'
    else:
        d_s = '  —'
    print(f'  {nombre:<22} {tipo:<15} {nivel:<7} {f1_s:>9} {s1_s:>6} {f2_s:>9} {s2_s:>6} {d_s:>7}')

print(SEP)
if FASE2:
    best2 = max(FASE2, key=lambda m: m['f1'] if not np.isnan(m['f1']) else -1)
    print(f'\n  Mejor Fase 2: {best2["nombre"]}  F1={best2["f1"]:.4f} ± {best2["std"]:.4f}')

---
## 3. Evolución Fase 1 → Fase 2

In [ ]:
# ── Celda 4: Gráfica de evolución ────────────────────────────────────────────
comunes = [n for n in ORDEN
           if n in f1_f1 and n in f1_f2
           and not np.isnan(f1_f1[n]) and not np.isnan(f1_f2[n])]

if not comunes:
    print('⚠️  No hay modelos comunes entre Fase 1 y Fase 2 para graficar.')
else:
    comunes_sorted = sorted(comunes, key=lambda n: f1_f2[n], reverse=True)
    v1 = [f1_f1[n] for n in comunes_sorted]
    v2 = [f1_f2[n] for n in comunes_sorted]
    s1 = [std_f1.get(n, 0) for n in comunes_sorted]
    s2 = [std_f2.get(n, 0) for n in comunes_sorted]
    cols = [COLORES.get(n, '#94A3B8') for n in comunes_sorted]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Panel izq: barras agrupadas
    ax = axes[0]
    x = np.arange(len(comunes_sorted))
    w = 0.35
    b1 = ax.bar(x - w/2, v1, w, yerr=s1, capsize=4,
                label='Fase 1 (2-Fold)', color=[c+'99' for c in cols],
                edgecolor='white', error_kw={'linewidth':1.2})
    b2 = ax.bar(x + w/2, v2, w, yerr=s2, capsize=4,
                label='Fase 2 (5-Fold)', color=cols,
                edgecolor='white', error_kw={'linewidth':1.2})
    ax.set_xticks(x)
    ax.set_xticklabels([n.replace(' ', '\n') for n in comunes_sorted], fontsize=8)
    ax.set_ylabel('F1-Score', fontsize=11)
    ax.set_ylim(0, 1.08)
    ax.set_title('F1-Score por modelo y fase', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            if h and h > 0.01:
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.012,
                        f'{h:.3f}', ha='center', va='bottom', fontsize=7)

    # Panel der: slope chart
    ax2 = axes[1]
    for i, n in enumerate(comunes_sorted):
        c = COLORES.get(n, '#94A3B8')
        ax2.plot([0, 1], [f1_f1[n], f1_f2[n]], 'o-', color=c, linewidth=2,
                 markersize=8, label=n)
        ax2.text(-0.08, f1_f1[n], f'{f1_f1[n]:.3f}', ha='right', va='center',
                 fontsize=8, color=c)
        ax2.text(1.08, f1_f2[n], f'{f1_f2[n]:.3f}', ha='left', va='center',
                 fontsize=8, color=c)
    ax2.set_xlim(-0.35, 1.35)
    ax2.set_xticks([0, 1])
    ax2.set_xticklabels(['Fase 1\n(2-Fold)', 'Fase 2\n(5-Fold)'], fontsize=11)
    ax2.set_ylabel('F1-Score', fontsize=11)
    ax2.set_title('Evolución por modelo', fontsize=12, fontweight='bold')
    ax2.legend(loc='lower right', fontsize=8, framealpha=0.7)
    ax2.grid(axis='y', linestyle='--', alpha=0.3)

    plt.suptitle('Landslide4Sense — Evolución del proyecto', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'evolucion_fase1_fase2.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Guardado: evolucion_fase1_fase2.png')

---
## 4. Variabilidad por fold — Box plots (Fase 2)

In [ ]:
# ── Celda 5: Box plots Fase 2 ─────────────────────────────────────────────────
mod_folds = [m for m in FASE2 if len(m['folds_f1']) >= 2 and not any(np.isnan(m['folds_f1']))]

if not mod_folds:
    print('⚠️  No hay datos de folds para box plots (Fase 2).')
else:
    mod_folds_s = sorted(mod_folds, key=lambda m: m['f1'], reverse=True)
    data  = [m['folds_f1']  for m in mod_folds_s]
    noms  = [m['nombre']    for m in mod_folds_s]
    cols  = [COLORES.get(m['nombre'], '#94A3B8') for m in mod_folds_s]

    fig, ax = plt.subplots(figsize=(12, 5))
    bp = ax.boxplot(data, patch_artist=True, notch=False, widths=0.5,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], cols):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    rng = np.random.default_rng(42)
    for i, (d, c) in enumerate(zip(data, cols), 1):
        jitter = rng.uniform(-0.12, 0.12, len(d))
        ax.scatter([i + j for j in jitter], d, color=c, edgecolor='white',
                   zorder=3, s=55, alpha=0.9)

    ax.set_xticks(range(1, len(noms)+1))
    ax.set_xticklabels([n.replace(' ', '\n') for n in noms], fontsize=9)
    ax.set_ylabel('F1-Score (por fold)', fontsize=11)
    ax.set_title('Variabilidad entre folds — Fase 2 (5-Fold CV)', fontsize=12, fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'boxplots_fase2.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Guardado: boxplots_fase2.png')

---
## 5. Tests estadísticos — Friedman + Wilcoxon (Fase 2)

In [ ]:
# ── Celda 6: Tests estadísticos ───────────────────────────────────────────────
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations

m5 = [m for m in FASE2
      if len(m['folds_f1']) == 5
      and not any(np.isnan(m['folds_f1']))]

if len(m5) < 2:
    print('⚠️  Se necesitan ≥2 modelos con 5 folds para los tests.')
else:
    grupos = [m['folds_f1'] for m in m5]
    stat_f, p_f = friedmanchisquare(*grupos)
    print('='*62)
    print(f'  TEST DE FRIEDMAN  ({len(m5)} modelos × 5 folds)')
    print('='*62)
    print(f'  χ² = {stat_f:.4f}  |  p = {p_f:.4f}')
    if p_f < 0.05:
        print('  → Diferencias SIGNIFICATIVAS entre modelos (p < 0.05) ✅')
    else:
        print('  → No se detectan diferencias significativas (p ≥ 0.05)')

    print()
    print('  TESTS DE WILCOXON pareados (mejor vs cada uno):')
    print('  ' + '-'*56)
    mejor = max(m5, key=lambda m: m['f1'])
    print(f'  Referencia: {mejor["nombre"]}  (F1={mejor["f1"]:.4f})')
    print()
    for m in m5:
        if m['nombre'] == mejor['nombre']: continue
        try:
            stat_w, p_w = wilcoxon(mejor['folds_f1'], m['folds_f1'])
            sig = '✅ sig.' if p_w < 0.05 else '  n.s. '
            print(f'  {mejor["nombre"]} vs {m["nombre"]:<22}  p={p_w:.4f}  {sig}')
        except Exception as e:
            print(f'  {m["nombre"]:<28} → no se pudo calcular ({e})')
    print('='*62)

---
## 6. Benchmarking vs literatura

In [ ]:
# ── Celda 7: Comparación con literatura ──────────────────────────────────────
# Referencias sobre Landslide4Sense (actualiza si tienes más recientes)
LITERATURA = [
    {'ref': 'Youssef et al. 2021',  'f1': 0.870, 'metrica': 'F1 patch',  'notas': 'RF + features L4S'},
    {'ref': 'Song et al. 2025',     'f1': 0.780, 'metrica': 'F1 píxel',  'notas': 'Transformer + U-Net'},
    {'ref': 'Wang et al. 2024',     'f1': 0.740, 'metrica': 'F1 píxel',  'notas': 'Encoder dual SAR+Óptico'},
    {'ref': 'Ghorbanzadeh 2022',    'f1': 0.720, 'metrica': 'F1 píxel',  'notas': 'U-Net baseline L4S'},
]

fig, ax = plt.subplots(figsize=(13, 6))
y_pos = 0
yticks, ylabels = [], []

# Nuestros modelos Fase 2 (ordenados)
mod_sorted = sorted(FASE2, key=lambda m: m['f1'] if not np.isnan(m['f1']) else -1, reverse=True)
for m in mod_sorted:
    if np.isnan(m['f1']): continue
    c = COLORES.get(m['nombre'], '#94A3B8')
    ax.barh(y_pos, m['f1'], xerr=m['std'], color=c, edgecolor='white',
            capsize=4, height=0.6, alpha=0.85)
    ax.text(m['f1'] + m['std'] + 0.005, y_pos, f"{m['f1']:.3f}",
            va='center', fontsize=8)
    yticks.append(y_pos)
    ylabels.append(f"{m['nombre']} (Fase 2)")
    y_pos += 1

y_pos += 0.5  # separador

# Literatura
for lit in sorted(LITERATURA, key=lambda x: x['f1'], reverse=True):
    ax.barh(y_pos, lit['f1'], color='#CBD5E1', edgecolor='#94A3B8',
            height=0.6, alpha=0.7, hatch='//')
    ax.text(lit['f1'] + 0.005, y_pos, f"{lit['f1']:.3f}  [{lit['ref']}]",
            va='center', fontsize=8, color='#475569')
    yticks.append(y_pos)
    ylabels.append(f"{lit['notas']}")
    y_pos += 1

ax.set_yticks(yticks)
ax.set_yticklabels(ylabels, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('F1-Score', fontsize=11)
ax.set_xlim(0, 1.05)
ax.set_title('Nuestros modelos (Fase 2) vs Literatura — Landslide4Sense',
             fontsize=12, fontweight='bold')
ax.axvline(0.7, color='#EF4444', linewidth=1, linestyle=':', alpha=0.6, label='Umbral competitivo 0.70')
ax.legend(fontsize=9)
ax.grid(axis='x', linestyle='--', alpha=0.3)

# Parche leyenda
p1 = mpatches.Patch(color='#818CF8', label='Nuestros modelos (Fase 2)')
p2 = mpatches.Patch(facecolor='#CBD5E1', hatch='//', edgecolor='#94A3B8', label='Literatura')
ax.legend(handles=[p1, p2], loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'benchmarking_literatura.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: benchmarking_literatura.png')

---
## 7. Transferibilidad a Colombia

In [ ]:
# ── Celda 8: Transferibilidad Colombia ───────────────────────────────────────
# El JSON lo genera la última celda de transferibilidad_01_ablacion_robustez.ipynb.

if _col and "resultados_rf" in _col:
    res_rf   = _col["resultados_rf"]    # {nombre: [mean_f1, std_f1]}
    res_unet = _col.get("res_unet", {})  # {nombre: f1_valor}
    print(f"✅ Colombia — {len(res_rf)} grupos RF, {len(res_unet)} grupos U-Net")
else:
    print("⚠️  JSON de Colombia no encontrado.")
    print("   Ejecuta la última celda de transferibilidad_01 y vuelve a correr esto.")
    res_rf   = {"Sin datos": [0.0, 0.0]}
    res_unet = {}

# Mapeo de etiquetas para la gráfica
LABEL_MAP = {
    "Linea base (14 ch)": "Base (14ch)",
    "Linea base (14ch)":  "Base (14ch)",
}
GRUPOS_NOMBRES = list(res_rf.keys())
labels_graf = [LABEL_MAP.get(k, k.replace("Sin ", "Sin ")) for k in GRUPOS_NOMBRES]
rf_f1   = [v[0] if isinstance(v, (list, tuple)) else v for v in res_rf.values()]
unet_f1 = [res_unet.get(k, 0) or 0 for k in GRUPOS_NOMBRES]

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(GRUPOS_NOMBRES))
w = 0.35
bars_rf = ax.bar(x - w/2, rf_f1, w, label="RF (patch-level)",
                 color="#2E5FA3", edgecolor="white")
bars_un = ax.bar(x + w/2, unet_f1, w, label="U-Net (pixel-level)",
                 color="#D97706", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(labels_graf, fontsize=9)
ax.set_ylabel("F1-Score", fontsize=11)
ax.set_title("Ablacion por grupo de sensores — Transferibilidad a Colombia",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
ax.grid(axis="y", linestyle="--", alpha=0.3)
for bar in list(bars_rf) + list(bars_un):
    h = bar.get_height()
    if h and h > 0.01:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.3f}",
                ha="center", va="bottom", fontsize=7.5)
plt.tight_layout()
plt.savefig(OUT_DIR / "transferibilidad_colombia.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: transferibilidad_colombia.png")

---
## 8. Gap analysis — Radar chart

In [ ]:
# ── Celda 9: Radar chart — posición vs SOTA ───────────────────────────────────
DIMENSIONES = [
    'Protocolo\nevaluación',
    'Arquitectura\nmodelo',
    'Augmentación\ndatos',
    'Análisis\nerror',
    'Test set\noficial',
    'Ensamble\nmodelos',
    'Fusión\nSAR/Óptico',
    'Explicabilidad',
]
NUESTRO  = [0.9, 0.5, 0.5, 0.7, 0.1, 0.2, 0.3, 0.7]
SOTA_REF = [1.0, 1.0, 0.9, 0.8, 1.0, 0.8, 0.9, 0.7]

N = len(DIMENSIONES)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
# Cerrar el círculo
NUESTRO_c  = NUESTRO  + [NUESTRO[0]]
SOTA_c      = SOTA_REF + [SOTA_REF[0]]
angles_c    = angles   + [angles[0]]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.plot(angles_c, SOTA_c,  'o-', linewidth=2, color='#6B7280', label='SOTA referencia', alpha=0.7)
ax.fill(angles_c, SOTA_c,  alpha=0.08, color='#6B7280')
ax.plot(angles_c, NUESTRO_c, 'o-', linewidth=2.5, color='#2E5FA3', label='Nuestro proyecto')
ax.fill(angles_c, NUESTRO_c, alpha=0.2, color='#2E5FA3')

ax.set_thetagrids(np.degrees(angles), DIMENSIONES, fontsize=9)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=7)
ax.set_title('Gap Analysis — Nuestro proyecto vs SOTA', fontsize=12,
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)

plt.tight_layout()
plt.savefig(OUT_DIR / 'gap_analysis_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: gap_analysis_radar.png')

---
## 9. Resumen ejecutivo

In [ ]:
# ── Celda 10: Resumen ejecutivo ───────────────────────────────────────────────
SOTA_F1_PIXEL = 0.780  # Song et al. 2025 (Transformer+UNet)
SOTA_F1_PATCH = 0.870  # Youssef et al. 2021

best1 = max(FASE1, key=lambda m: m['f1'] if not np.isnan(m['f1']) else -1) if FASE1 else None
best2 = max(FASE2, key=lambda m: m['f1'] if not np.isnan(m['f1']) else -1) if FASE2 else None

SEP = '=' * 66
print(SEP)
print('  RESUMEN EJECUTIVO — DETECCIÓN DE DESLIZAMIENTOS L4S')
print(SEP)

print('\n  FASE 1 — Análisis Inicial (2-Fold, ~2 000 muestras)')
print('  ' + '-'*60)
if best1:
    print(f'  Mejor modelo : {best1["nombre"]:<22}  F1 = {best1["f1"]:.4f} ± {best1["std"]:.4f}')
    print(f'  Protocolo    : {best1["n_folds"]}-Fold · nivel {best1["nivel"]}')
for m in sorted(FASE1, key=lambda x: x['f1'] if not np.isnan(x['f1']) else -1, reverse=True):
    if np.isnan(m['f1']): continue
    print(f'    {m["nombre"]:<22} F1={m["f1"]:.4f}  (±{m["std"]:.4f})')

print('\n  FASE 2 — Protocolo Comparable Literatura (5-Fold, 3 799 muestras)')
print('  ' + '-'*60)
if best2:
    print(f'  Mejor modelo : {best2["nombre"]:<22}  F1 = {best2["f1"]:.4f} ± {best2["std"]:.4f}')
    print(f'  Protocolo    : 5-Fold CV · nivel {best2["nivel"]}')
for m in sorted(FASE2, key=lambda x: x['f1'] if not np.isnan(x['f1']) else -1, reverse=True):
    if np.isnan(m['f1']): continue
    print(f'    {m["nombre"]:<22} F1={m["f1"]:.4f}  (±{m["std"]:.4f})')

print('\n  ESTADO DEL ARTE (referencia)')
print('  ' + '-'*60)
print(f'  F1 píxel SOTA : {SOTA_F1_PIXEL:.4f}  (Song et al. 2025, Transformer+U-Net)')
print(f'  F1 patch SOTA : {SOTA_F1_PATCH:.4f}  (Youssef et al. 2021, RF+features)')
if best2:
    nivel_sota = SOTA_F1_PIXEL if best2['nivel']=='píxel' else SOTA_F1_PATCH
    gap = (nivel_sota - best2['f1']) * 100
    print(f'\n  Gap vs SOTA   : {gap:+.1f} pp  ({best2["nombre"]} vs referencia {best2["nivel"]})')

if FASE1 and FASE2 and best1 and best2:
    delta = (best2['f1'] - best1['f1']) * 100
    print(f'  Mejora F1     : {delta:+.1f} pp  (Fase 1 → Fase 2, mejor modelo cada fase)')

print('\n  TRANSFERIBILIDAD')
print('  ' + '-'*60)
if _col:
    print('  Ablación Colombia completada — ver notebook transferibilidad_01')
else:
    print('  Pendiente: correr transferibilidad_01_ablacion_robustez.ipynb')

print('\n  BRECHAS CLAVE vs SOTA')
print('  ' + '-'*60)
print('  • Evaluación en test set oficial L4S no realizada')
print('  • Augmentación avanzada (mix-up, elastic) no implementada')
print('  • Ensamble de modelos no explorado')
print('  • Fusión SAR+Óptico (encoder dual) no implementada')
print()
print(SEP)
print('  Figuras guardadas en:', OUT_DIR)
print(SEP)